In [65]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("../data/processed/cleaned_apple_health.csv")

In [66]:
# Ensure datetimes are parsed (your sample shows timezone offsets like -04:00)
for col in ["start", "end"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

# Filter glucose rows — either by rtype or your 'signal' column
glucose = df[(df["rtype"] == "BloodGlucose")].copy()

# Coerce 'value' to float (your unit looks like "mmol<...>/L"; value may be string)
def to_float(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    m = re.search(r"[-+]?\d*\.?\d+", str(x))
    return float(m.group()) if m else np.nan

glucose["mmol_L"] = glucose["value"].apply(to_float)

# Convert mmol/L → mg/dL (use 18.0 to keep it simple)
glucose["mg_dL"] = glucose["mmol_L"] * 18.0

# Use 'start' as the timestamp
g = glucose[["start", "mg_dL"]].dropna().drop_duplicates(subset=["start"]).set_index("start").sort_index()

# Resample to 5-minute grid and interpolate (linear is fine for baselines)
g5 = g.resample("5T").mean().interpolate("time", limit=3)   # limit=3 avoids long FF fill

# Optional: clip to plausible CGM range
g5["mg_dL"] = g5["mg_dL"].clip(lower=39, upper=301)

# Save for reuse
g5 = g5.rename(columns={"mg_dL":"glucose_mgdl"})
g5.to_csv("processed_glucose_5min.csv")
print(g5.head())

                           glucose_mgdl
start                                  
2025-07-28 23:00:00+00:00      74.93508
2025-07-28 23:05:00+00:00      83.92734
2025-07-28 23:10:00+00:00      88.92306
2025-07-28 23:15:00+00:00      95.91696
2025-07-28 23:20:00+00:00      88.92306


/var/folders/sk/hr3tvvss7yxf6t4ckk550zn80000gn/T/ipykernel_52830/33414461.py:27: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  g5 = g.resample("5T").mean().interpolate("time", limit=3)   # limit=3 avoids long FF fill


In [67]:
circ = g5.copy()
circ["tod_min"] = circ.index.tz_convert("UTC").hour*60 + circ.index.tz_convert("UTC").minute
circ_profile = (circ.groupby("tod_min")["glucose_mgdl"].median()).reindex(range(0,24*60), method="nearest")
def circadian_baseline(ts):
    idx = (ts.hour*60 + ts.minute).astype(int)
    return circ_profile.iloc[idx].values

### --- 3) Extract DIET rows and group into meals ---

In [68]:
is_diet_by_signal = df.get("signal","").astype(str).str.startswith("diet_", na=False)
is_diet_by_rtype  = df["rtype"].astype(str).str.contains("Dietary", na=False)
diet = df[is_diet_by_signal | is_diet_by_rtype].copy()


In [69]:
#  Normalize a "nutrient" name
def nutrient_name(row):
    if pd.notna(row.get("signal")) and str(row["signal"]).startswith("diet_"):
        return str(row["signal"]).replace("diet_","")
    # HK identifiers fallback
    rt = str(row["rtype"])
    if "Dietary" in rt:
        return rt.split("Dietary")[-1].lower()  # e.g., 'Carbohydrates' -> 'carbohydrates'
    return "unknown"

In [70]:
diet["nutrient"] = diet.apply(nutrient_name, axis=1)
diet["ts"] = diet["start"].dt.floor("1min")

# Group rows that are within 15 minutes as one meal
diet = diet.sort_values("ts")
gap_min = (diet["ts"] - diet["ts"].shift()).dt.total_seconds().div(60).fillna(9999)
diet["meal_id"] = (gap_min > 15).cumsum()
meals_wide = (diet.pivot_table(index="meal_id", columns="nutrient", values="value", aggfunc="sum")
                   .fillna(0))
meal_time = diet.groupby("meal_id")["ts"].min().rename("meal_time")
meals = meals_wide.join(meal_time)

In [71]:
# Keep common nutrients; compute net carbs
for col in ["carbohydrates","fiber","sugar","fattotal","protein","energyconsumed"]:
    if col not in meals: meals[col] = 0.0
meals["carbs"]     = meals["carbohydrates"]
meals["fiber"]     = meals["fiber"]
meals["sugar"]     = meals["sugar"]
meals["fat"]       = meals["fattotal"]
meals["protein"]   = meals["protein"]
meals["kcal"]      = meals["energyconsumed"]
meals["net_carbs"] = (meals["carbs"] - meals["fiber"]).clip(lower=0)

In [72]:
# --- 4) Build targets using CGM (calibration only) ---
H = 120  # minutes post-meal window
pre = 30 # minutes baseline window
index5 = g5.index


In [73]:
def window(ts, m):
    return (ts, ts + pd.Timedelta(minutes=m))

rows = []
for mid, row in meals.iterrows():
    t0 = row["meal_time"]
    if pd.isna(t0): continue
    # baseline: mean of [t0-30, t0)
    start_b, end_b = t0 - pd.Timedelta(minutes=pre), t0
    base = g5.loc[start_b:end_b]["glucose_mgdl"].mean()
    # future window: [t0, t0+H]
    start_f, end_f = window(t0, H)
    future = g5.loc[start_f:end_f]["glucose_mgdl"]
    if base is np.nan or future.empty: continue
    peak = future.max()
    auc  = (future - base).clip(lower=0).sum() * 5.0/60.0  # mg/dL·hour
    rows.append({
        "meal_id": mid, "meal_time": t0,
        "base": base, "peak": peak, "d_peak": peak - base, "auc_0_2h": auc,
        "carbs": row["carbs"], "net_carbs": row["net_carbs"], "fiber": row["fiber"],
        "sugar": row["sugar"], "fat": row["fat"], "protein": row["protein"], "kcal": row["kcal"]
    })

dataset = pd.DataFrame(rows).sort_values("meal_time")

In [74]:
# Add context features available at t0 (no leakage)
dataset["hour"] = dataset["meal_time"].dt.hour
dataset["dow"]  = dataset["meal_time"].dt.weekday
dataset["sin_hour"] = np.sin(2*np.pi*dataset["hour"]/24)
dataset["cos_hour"] = np.cos(2*np.pi*dataset["hour"]/24)


In [75]:
def series_5min_from_rtype(df, rtype, how="mean"):
    s = df.loc[df["rtype"] == rtype, ["start", "value"]].copy()
    if s.empty:
        return pd.Series(dtype=float)

    s = (s.set_index("start")
           .sort_index()
           .resample("5min")
           .agg(how))

    # if resample returns a DataFrame, squeeze to Series
    if isinstance(s, pd.DataFrame):
        s = s["value"]

    # interpolate small gaps
    s = s.interpolate("time", limit=3)
    s.name = rtype
    return s

def attach_past_feature(meals_df, series_5min, minutes, how, colname):
    """Compute past-{minutes} {how} of a series at each meal time (no leakage)."""
    if series_5min.empty:
        meals_df[colname] = np.nan
        return meals_df
    # align meal times to 5-min grid to avoid tolerance issues
    mt = meals_df["meal_time"].dt.floor("5min")
    roll = series_5min.rolling(f"{minutes}min").agg(how)
    # ensure index has the meal timestamps (if missing, reindex with nearest within 2.5 min)
    vals = roll.reindex(mt, method="nearest", tolerance=pd.Timedelta(minutes=2.5))
    meals_df[colname] = vals.values
    return meals_df

In [76]:
# ========= 4) Build time-of-day features (cyclical) =========
meals["hour"] = meals["meal_time"].dt.hour
meals["dow"]  = meals["meal_time"].dt.weekday
meals["sin_hour"] = np.sin(2*np.pi*meals["hour"]/24)
meals["cos_hour"] = np.cos(2*np.pi*meals["hour"]/24)

hr_5   = series_5min_from_rtype(df, "HeartRate",              how="mean")
aee_5  = series_5min_from_rtype(df, "ActiveEnergyBurned",     how="mean")  # Apple often logs per-minute; we’ll sum in the past-window step
steps5 = series_5min_from_rtype(df, "StepCount",              how="sum")
dist5  = series_5min_from_rtype(df, "DistanceWalkingRunning", how="sum")
rhr5   = series_5min_from_rtype(df, "RestingHeartRate",       how="mean")
ex5    = series_5min_from_rtype(df, "AppleExerciseTime",      how="sum")
fl5    = series_5min_from_rtype(df, "FlightsClimbed",         how="sum")

def attach_past_feature(meals_df, series_5min, minutes, how, colname):
    if series_5min.empty:
        meals_df[colname] = np.nan
        return meals_df
    mt = meals_df["meal_time"].dt.floor("5min")
    roll = series_5min.rolling(f"{minutes}min").agg(how)
    vals = roll.reindex(mt, method="nearest", tolerance=pd.Timedelta(minutes=2.5))
    meals_df[colname] = vals.values
    return meals_df

meals = attach_past_feature(meals, hr_5,   minutes=30,  how="mean", colname="hr_mean_30m")
meals = attach_past_feature(meals, aee_5,  minutes=60,  how="sum",  colname="active_kcal_60m")
meals = attach_past_feature(meals, steps5, minutes=60,  how="sum",  colname="steps_60m")
meals = attach_past_feature(meals, dist5,  minutes=60,  how="sum",  colname="walk_run_dist_60m")
meals = attach_past_feature(meals, rhr5,   minutes=1440,how="mean", colname="rest_hr_daily")
meals = attach_past_feature(meals, ex5,    minutes=60,  how="sum",  colname="exercise_min_60m")
meals = attach_past_feature(meals, fl5,    minutes=60,  how="sum",  colname="flights_60m")


In [77]:
# ========= 6) Build targets from CGM for calibration (personalized) =========
H   = 120  # minutes after meal
pre = 30   # minutes before
rows = []
for mid, row in meals.iterrows():
    t0 = row["meal_time"]
    if pd.isna(t0): continue
    base = g5.loc[t0 - pd.Timedelta(minutes=pre): t0]["glucose_mgdl"].mean()
    future = g5.loc[t0 : t0 + pd.Timedelta(minutes=H)]["glucose_mgdl"]
    if np.isnan(base) or future.empty: 
        continue
    peak = future.max()
    auc  = (future - base).clip(lower=0).sum() * 5/60  # mg/dL·hour
    rows.append({
        "meal_id": mid, "meal_time": t0, "base": base, "peak": peak,
        "d_peak": peak - base, "auc_0_2h": auc
    })

targets = pd.DataFrame(rows).set_index("meal_id")

In [78]:
targets = pd.DataFrame(rows).set_index("meal_id")
targets = pd.DataFrame(rows).set_index("meal_id")
targets = targets.drop(columns=["meal_time"], errors="ignore")  # meals already has it

# ========= 7) Final dataset for modeling =========
dataset = meals.join(targets, how="inner").reset_index(drop=False)

# Keep the columns you want for modeling:
feature_cols = [
    "net_carbs","carbs","fiber","fat","protein","sugar","kcal",
    "sin_hour","cos_hour","dow",
    "hr_mean_30m","active_kcal_60m","steps_60m","walk_run_dist_60m",
    "rest_hr_daily","exercise_min_60m","flights_60m",
]
target_cols = ["d_peak","auc_0_2h"]

print(dataset[["meal_time"] + feature_cols + target_cols].head())

                  meal_time  net_carbs     carbs     fiber      fat   protein  \
0 2025-07-30 12:00:00+00:00   80.11570  95.36570  15.25000  46.8651  75.49040   
1 2025-07-30 13:37:00+00:00   35.42055  39.69750   4.27695  16.8392  27.50140   
2 2025-07-31 11:52:00+00:00   51.30057  63.92252  12.62195  49.4770  58.46362   
3 2025-08-01 13:09:00+00:00   29.25370  36.00670   6.75300  14.9160  23.78660   
4 2025-08-01 13:27:00+00:00    1.62900   3.87900   2.25000   8.9874   3.80700   

      sugar      kcal      sin_hour  cos_hour  dow  hr_mean_30m  \
0  14.08500  1076.849  1.224647e-16 -1.000000    2    97.584852   
1  15.30210   408.376 -2.588190e-01 -0.965926    2          NaN   
2  25.40512   919.279  2.588190e-01 -0.965926    3   118.583338   
3  16.02100   359.890 -2.588190e-01 -0.965926    4    71.999450   
4   0.78300   104.220 -2.588190e-01 -0.965926    4    77.499450   

   active_kcal_60m  steps_60m  walk_run_dist_60m  rest_hr_daily  \
0         3.874726     1172.0           0.4

In [ ]:
# Save as CSV
dataset.to_csv("../data/processed/processed_meal_dataset.csv", index=False)

# Or save as Parquet (faster, smaller, preserves dtypes)
dataset.to_parquet("../data/processed/processed_meal_dataset.parquet", index=False)